# 03 · Label train/test with an LLM, fine-tune RoBERTa (16-label system, v1)

**Step 8.** Label the 16,328-review train set and 4,083-review test set with GPT-4.1-mini
(3 runs + majority vote, with a GPT-4o-mini fallback).

**Step 9.** Fine-tune RoBERTa for multi-label classification over **20 classes**
(16 emotions + 4 evaluation dimensions) and compare four configurations:

| config | change |
|---|---|
| `config1_baseline` | roberta-large, 2e-5 |
| `config2_higher_lr` | 5e-5 |
| `config3_roberta_base` | smaller model |
| `config4_long_context` | 256 tokens |

Everything is scored on the human-labeled holdout. A threshold sweep on config 1 follows.

**Takeaway:** best macro F1 was **0.600**. Rare emotions such as Sadness and Guilt scored 0.00 and dragged the
macro average down, which motivated the redesign in notebook 04.

**Needs:** a GPU, `OPENAI_API_KEY`, the files from notebooks 01 to 02, and the human-labeled holdout.


In [ ]:
# ── Setup (runs locally or on Colab) ─────────────────────────────────────────
# Install dependencies first:  pip install -r requirements.txt
import os

# Where datasets, LLM labels, checkpoints and models are written (not committed to git).
# Colab: mount Drive and set CAPSTONE_DATA_DIR to a Drive folder to persist between sessions.
DATA_DIR = os.environ.get("CAPSTONE_DATA_DIR", "data")
os.makedirs(DATA_DIR, exist_ok=True)
OUTPUT_DIR = DATA_DIR

# API keys: environment variables first, Colab Secrets as a fallback.
_ENV_NAMES = {
    "claude": "ANTHROPIC_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "xai": "XAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "minimax": "MINIMAX_API_KEY",
}

def get_api_key(name):
    value = os.environ.get(_ENV_NAMES.get(name, name)) or os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata as _colab_userdata
        return _colab_userdata.get(name)
    except Exception:
        return None

class userdata:  # drop-in for google.colab.userdata
    get = staticmethod(get_api_key)


In [ ]:
# Imports used by the labeling cells below
import json, time, datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
from tqdm import tqdm
from openai import OpenAI


In [ ]:
# ── CELL 1: Step 8 config ─────────────────────────────────────────────────────
TRAIN_DIR      = f"{OUTPUT_DIR}/train_llm_labels"
CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoints_train"
os.makedirs(TRAIN_DIR,      exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

PRIMARY_VENDOR  = "openai"
PRIMARY_MODEL   = "gpt-4.1-mini"
FALLBACK_VENDOR = "openai"
FALLBACK_MODEL  = "gpt-4o-mini"
RUNS            = [1, 2, 3]

SYSTEM_PROMPT = """Label this product review for a consumer psychology study.

TASK A — EMOTIONS (multi-label, binary 0/1):
Return 1 if clearly present, 0 if not. Only label with explicit evidence.
Joy, Love, Surprise, Pride, Gratitude, Contentment, Relief, Hope, Interest,
Anger, Frustration, Disappointment, Sadness, Fear, Guilt, Disgust

TASK B — EVALUATIONS (single label each):
Trust: Low / Medium / High
Commitment: Low / Medium / High
Recommendation: Would Not Recommend / Neutral / Would Recommend
Sentiment: Negative / Neutral / Positive

Rules: explicit evidence only, middle label if unclear, no skipping eval dims.

Return ONLY this JSON, nothing else:
{
  "emotions": {
    "Joy": 0, "Love": 0, "Surprise": 0, "Pride": 0, "Gratitude": 0,
    "Contentment": 0, "Relief": 0, "Hope": 0, "Interest": 0,
    "Anger": 0, "Frustration": 0, "Disappointment": 0,
    "Sadness": 0, "Fear": 0, "Guilt": 0, "Disgust": 0
  },
  "Trust": "Medium",
  "Commitment": "Medium",
  "Recommendation": "Neutral",
  "Sentiment": "Neutral"
}"""

print("✓ Step 8 config loaded")
print(f"  Primary  : {PRIMARY_VENDOR}/{PRIMARY_MODEL}")
print(f"  Fallback : {FALLBACK_VENDOR}/{FALLBACK_MODEL}")

✓ Step 8 config loaded
  Primary  : openai/gpt-4.1-mini
  Fallback : openai/gpt-4o-mini


In [ ]:
# ── CELL 2: Parse + cost ──────────────────────────────────────────────────────
def parse_response_full(text, used_vendor, used_model):
    try:
        text = text.strip()
        if "</think>" in text:
            text = text.split("</think>")[-1].strip()
        if "```" in text:
            lines = [l for l in text.split("\n") if not l.strip().startswith("```")]
            text  = "\n".join(lines).strip()

        start = text.find("{")
        if start == -1:
            return None, None, "No JSON found"

        depth, end = 0, start
        for i, ch in enumerate(text[start:], start):
            if ch == "{":   depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    end = i + 1
                    break
        if depth > 0:
            text = text + "}" * depth

        data = json.loads(text[start:end] if depth == 0 else text[start:])

        emotions_out = {}
        raw_em = data.get("emotions", {})
        for em in EMOTIONS:
            emotions_out[em] = int(bool(raw_em.get(em, 0)))

        eval_out = {}
        for dim, levels in EVAL_DIMS.items():
            val = data.get(dim, "")
            eval_out[dim] = val if val in levels else levels[len(levels) // 2]

        return emotions_out, eval_out, None

    except Exception as e:
        return None, None, str(e)

def compute_cost_full(vendor, model, input_tokens, output_tokens):
    prices = {
        ("openai", "gpt-4.1-mini"): (0.40, 1.60),
        ("openai", "gpt-4o-mini"):  (0.15, 0.60),
    }
    price_in, price_out = prices.get((vendor, model), (0.40, 1.60))
    return (input_tokens / 1e6) * price_in + (output_tokens / 1e6) * price_out

print("✓ Parse + cost ready")

✓ Parse + cost ready


In [ ]:
# ── CELL 3: API call + classify ───────────────────────────────────────────────
def call_openai_full(review_text, model, system_prompt):
    client = OpenAI(api_key=get_api_key("OPENAI_API_KEY"))
    resp   = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": review_text[:2000]}
        ],
        temperature=0,
        max_tokens=300,
        response_format={"type": "json_object"},
    )
    return resp.choices[0].message.content or "", resp.usage.prompt_tokens, resp.usage.completion_tokens

def classify_review_full(row_id, review_text, retries=3):
    wait_times  = [3, 6, 15]
    used_vendor = PRIMARY_VENDOR
    used_model  = PRIMARY_MODEL

    for attempt in range(retries):
        try:
            t0 = time.perf_counter()
            raw, in_tok, out_tok = call_openai_full(review_text, used_model, SYSTEM_PROMPT)
            latency = time.perf_counter() - t0

            emotions_out, eval_out, err = parse_response_full(raw, used_vendor, used_model)
            if err:
                raise ValueError(err)

            cost   = compute_cost_full(used_vendor, used_model, in_tok, out_tok)
            record = {
                "id":          row_id,
                "review_text": review_text,
                "error":       None,
                "cost_usd":    cost,
                "latency_s":   latency,
                "model_used":  used_model,
            }
            for em in EMOTIONS:
                record[em] = emotions_out.get(em, 0)
            for dim in EVAL_DIMS:
                record[dim] = eval_out.get(dim, EVAL_DIMS[dim][len(EVAL_DIMS[dim]) // 2])
            return record

        except Exception as e:
            err_str       = str(e)
            is_rate_limit = any(x in err_str.lower() for x in ["rate", "429", "ratelimit"])

            if is_rate_limit and used_model == PRIMARY_MODEL:
                print(f"  ⚠ Rate limit row {row_id}, falling back to {FALLBACK_MODEL}")
                used_vendor = FALLBACK_VENDOR
                used_model  = FALLBACK_MODEL
                time.sleep(5)
                continue

            if attempt < retries - 1:
                time.sleep(60 if is_rate_limit else wait_times[attempt])
            else:
                record = {
                    "id":          row_id,
                    "review_text": review_text,
                    "error":       err_str,
                    "cost_usd":    0,
                    "latency_s":   0,
                    "model_used":  used_model,
                }
                for em in EMOTIONS:
                    record[em] = 0
                for dim in EVAL_DIMS:
                    record[dim] = EVAL_DIMS[dim][len(EVAL_DIMS[dim]) // 2]
                return record

print("✓ classify_review_full() ready")

✓ classify_review_full() ready


In [ ]:
# ── CELL 4: run_dataset — one dataset, one run ────────────────────────────────
def run_dataset(dataset_name, csv_path, run=1, max_workers=20):
    base_name = f"{dataset_name}_gpt41mini_run{run}"
    ckpt_path = f"{CHECKPOINT_DIR}/{base_name}.pkl"
    run_dir   = f"{TRAIN_DIR}/Run{run}"
    out_path  = f"{run_dir}/{base_name}.pkl"
    os.makedirs(run_dir, exist_ok=True)

    df       = pd.read_csv(csv_path)
    df["id"] = df.index

    results, processed_ids = [], set()
    if os.path.exists(ckpt_path):
        try:
            ck            = pd.read_pickle(ckpt_path)
            ok            = ck[ck["error"].isna()]
            results       = ok.to_dict("records")
            processed_ids = set(ok["id"].values)
            print(f"  Resuming {dataset_name} run{run}: {len(results)} done, {len(df)-len(results)} remaining")
        except Exception as e:
            print(f"  Could not load checkpoint: {e}")
    else:
        print(f"  Starting fresh: {dataset_name} run{run} — {len(df)} reviews")

    df_todo = df[~df["id"].isin(processed_ids)].copy()
    lock    = __import__("threading").Lock()
    start   = datetime.datetime.now()

    rows = [row for _, row in df_todo.iterrows()]

    print(f"\n{'='*60}")
    print(f"  {dataset_name} | run {run} | workers: {max_workers}")
    print(f"  Model: {PRIMARY_MODEL} (fallback: {FALLBACK_MODEL})")
    print(f"  Reviews: {len(df)} total | {len(df_todo)} remaining")
    print(f"{'='*60}")

    completed = 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(classify_review_full, row["id"], row["review_text"]): row["id"] for row in rows}
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"{dataset_name} run{run}"):
            res = future.result()
            with lock:
                results.append(res)
                completed += 1
                if completed % 200 == 0:
                    pd.DataFrame(results).to_pickle(ckpt_path)
                    cost_so_far = sum(r["cost_usd"] for r in results if not r["error"])
                    elapsed     = (datetime.datetime.now() - start).total_seconds()
                    print(f"\n  [{completed}/{len(df_todo)}] checkpoint | cost: ${cost_so_far:.4f} | {elapsed/60:.1f}m elapsed")

    df_out = pd.DataFrame(results)
    df_out.to_pickle(ckpt_path)
    df_out.to_pickle(out_path)
    df_out.to_csv(out_path.replace(".pkl", ".csv"), index=False)

    ok         = df_out[df_out["error"].isna()]
    total_cost = ok["cost_usd"].sum()
    duration   = (datetime.datetime.now() - start).total_seconds()
    print(f"\n✓ {dataset_name} run {run} complete")
    print(f"  Successful : {len(ok)}/{len(df_out)}")
    print(f"  Cost       : ${total_cost:.4f}")
    print(f"  Runtime    : {duration:.0f}s ({duration/60:.1f}m)")
    return df_out

print("✓ run_dataset() ready")

✓ run_dataset() ready


In [ ]:
# ── CELL 5: majority vote ─────────────────────────────────────────────────────
def compute_majority_vote_final(dataset_name, runs=RUNS):
    run_dfs = {}
    for run in runs:
        path = f"{TRAIN_DIR}/Run{run}/{dataset_name}_gpt41mini_run{run}.pkl"
        if not os.path.exists(path):
            print(f"✗ Missing run {run}")
            return None
        df           = pd.read_pickle(path)
        run_dfs[run] = df[df["error"].isna()].set_index("id")

    common_ids = sorted(set.intersection(*[set(run_dfs[r].index) for r in runs]))
    print(f"Common reviews across {len(runs)} runs: {len(common_ids)}")

    results = []
    for row_id in tqdm(common_ids, desc=f"{dataset_name} majority vote"):
        rows   = {run: run_dfs[run].loc[row_id] for run in runs}
        record = {
            "id":          row_id,
            "review_text": rows[runs[0]]["review_text"],
        }
        for em in EMOTIONS:
            votes          = sum(int(rows[run].get(em, 0) or 0) for run in runs)
            record[em]     = 1 if votes > len(runs) / 2 else 0
            record[f"{em}_votes"] = votes
        for dim, levels in EVAL_DIMS.items():
            level_votes = {lv: 0 for lv in levels}
            for run in runs:
                val = rows[run].get(dim)
                if val in level_votes:
                    level_votes[val] += 1
            record[dim]            = max(level_votes, key=level_votes.get)
            record[f"{dim}_votes"] = level_votes
        results.append(record)

    df_mv = pd.DataFrame(results)
    out   = f"{TRAIN_DIR}/{dataset_name}_final_labeled.pkl"
    df_mv.to_pickle(out)
    df_mv.to_csv(out.replace(".pkl", ".csv"), index=False)

    print(f"\n✓ {dataset_name} majority vote complete: {len(df_mv)} reviews")
    print(f"\nLabel distribution:")
    for em in EMOTIONS:
        count = df_mv[em].sum()
        print(f"  {em:<15}: {count} ({count/len(df_mv)*100:.1f}%)")
    for dim in EVAL_DIMS:
        print(f"  {dim:<15}: {df_mv[dim].value_counts().to_dict()}")
    return df_mv

print("✓ compute_majority_vote_final() ready")

✓ compute_majority_vote_final() ready


In [ ]:
# ── CELL 6: EXECUTE ───────────────────────────────────────────────────────────
TRAIN_CSV = f"{OUTPUT_DIR}/train_final_v3.csv"
TEST_CSV  = f"{OUTPUT_DIR}/test_final_v3.csv"

# ── Train: all 3 runs then majority vote ──────────────────────────────────────
for run in RUNS:
    run_dataset("train", TRAIN_CSV, run=run, max_workers=20)

df_train_labeled = compute_majority_vote_final("train", runs=RUNS)

# ── Test: all 3 runs then majority vote ───────────────────────────────────────
for run in RUNS:
    run_dataset("test", TEST_CSV, run=run, max_workers=20)

df_test_labeled = compute_majority_vote_final("test", runs=RUNS)

print("\n✓ Step 8 complete")
print(f"  Train: {len(df_train_labeled)} labeled reviews → {TRAIN_DIR}/train_final_labeled.csv")
print(f"  Test : {len(df_test_labeled)} labeled reviews → {TRAIN_DIR}/test_final_labeled.csv")

  Resuming train run1: 16328 done, 0 remaining

  train | run 1 | workers: 20
  Model: gpt-4.1-mini (fallback: gpt-4o-mini)
  Reviews: 16328 total | 0 remaining


train run1: 0it [00:00, ?it/s]



✓ train run 1 complete
  Successful : 16328/16328
  Cost       : $6.7799
  Runtime    : 2s (0.0m)
  Resuming train run2: 16328 done, 0 remaining

  train | run 2 | workers: 20
  Model: gpt-4.1-mini (fallback: gpt-4o-mini)
  Reviews: 16328 total | 0 remaining


train run2: 0it [00:00, ?it/s]



✓ train run 2 complete
  Successful : 16328/16328
  Cost       : $6.7832
  Runtime    : 4s (0.1m)
  Resuming train run3: 10800 done, 5528 remaining

  train | run 3 | workers: 20
  Model: gpt-4.1-mini (fallback: gpt-4o-mini)
  Reviews: 16328 total | 5528 remaining


...


  [4000/4083] checkpoint | cost: $1.6609 | 11.5m elapsed


test run3: 100%|██████████| 4083/4083 [11:43<00:00,  5.80it/s]



✓ test run 3 complete
  Successful : 4083/4083
  Cost       : $1.6950
  Runtime    : 705s (11.8m)
Common reviews across 3 runs: 4083


test majority vote: 100%|██████████| 4083/4083 [00:01<00:00, 3135.09it/s]



✓ test majority vote complete: 4083 reviews

Label distribution:
  Joy            : 2001 (49.0%)
  Love           : 734 (18.0%)
  Surprise       : 254 (6.2%)
  Pride          : 63 (1.5%)
  Gratitude      : 207 (5.1%)
  Contentment    : 2204 (54.0%)
  Relief         : 554 (13.6%)
  Hope           : 446 (10.9%)
  Interest       : 542 (13.3%)
  Anger          : 209 (5.1%)
  Frustration    : 1231 (30.1%)
  Disappointment : 1582 (38.7%)
  Sadness        : 25 (0.6%)
  Fear           : 61 (1.5%)
  Guilt          : 12 (0.3%)
  Disgust        : 171 (4.2%)
  Trust          : {'High': 1819, 'Medium': 1541, 'Low': 723}
  Commitment     : {'Medium': 1938, 'High': 1221, 'Low': 924}
  Recommendation : {'Would Recommend': 1707, 'Neutral': 1538, 'Would Not Recommend': 838}
  Sentiment      : {'Positive': 2024, 'Neutral': 1160, 'Negative': 899}

✓ Step 8 complete
  Train: 16328 labeled reviews → data/train_llm_labels/train_final_labeled.csv
  Test : 4083 labeled reviews → data/train_llm_labels/test_fin

In [ ]:
# ── CELL 1: Install + Imports ─────────────────────────────────────────────────

import os
import json
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import (
    f1_score, roc_auc_score, matthews_corrcoef,
    precision_score, recall_score, hamming_loss,
    jaccard_score, confusion_matrix,
)
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = DATA_DIR
TRAIN_DIR  = f"{OUTPUT_DIR}/train_llm_labels"

EMOTIONS = [
    "Joy", "Love", "Surprise", "Pride", "Gratitude",
    "Contentment", "Relief", "Hope", "Interest",
    "Anger", "Frustration", "Disappointment",
    "Sadness", "Fear", "Guilt", "Disgust"
]
EVAL_DIMS = ["Trust", "Commitment", "Recommendation", "Sentiment"]
CLASSES   = EMOTIONS + EVAL_DIMS
NUM_LABELS = len(CLASSES)

EVAL_BINARIZE = {
    "Trust":          {"High": 1, "Medium": 0, "Low": 0},
    "Commitment":     {"High": 1, "Medium": 0, "Low": 0},
    "Recommendation": {"Would Recommend": 1, "Neutral": 0, "Would Not Recommend": 0},
    "Sentiment":      {"Positive": 1, "Neutral": 0, "Negative": 0},
}

print("✓ Imports ready")
print(f"  Labels: {NUM_LABELS} ({len(EMOTIONS)} emotions + {len(EVAL_DIMS)} eval dims)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 41.7 MB/s eta 0:00:00
✓ Imports ready
  Labels: 20 (16 emotions + 4 eval dims)


In [ ]:
# ── CELL 2: Load + Prepare Data (run once, reused across all configs) ──────────
print("="*60)
print("LOADING DATA")
print("="*60)

df_train_full = pd.read_csv(f"{TRAIN_DIR}/train_final_labeled.csv")
df_test       = pd.read_csv(f"{TRAIN_DIR}/test_final_labeled.csv")
df_holdout    = pd.read_csv(f"{OUTPUT_DIR}/final_holdout.csv")

# Clean holdout column names
emotion_rename = {
    "Joy (0/1)":             "Joy",
    "Love (0/1) ":           "Love",
    "Surprise (0/1)":        "Surprise",
    "Pride (0/1)":           "Pride",
    "Gratitude (0/1)":       "Gratitude",
    "Contentment (0/1)":     "Contentment",
    "Relief (0/1)":          "Relief",
    "Hope (0/1)":            "Hope",
    "Interest (0/1)":        "Interest",
    "Anger (0/1)":           "Anger",
    "Frustration (0/1)":     "Frustration",
    "Disappointment (0/1)":  "Disappointment",
    "Sadness (0/1)":         "Sadness",
    "Fear (0/1)":            "Fear",
    "Guilt (0/1)":           "Guilt",
    "Disgust (0/1)":         "Disgust",
}
eval_rename = {
    "Trust (Low / Medium / High)":                                             "Trust",
    "Commitment (Low / Medium / High)":                                        "Commitment",
    "Recommendation Intent (Would Not Recommend / Neutral / Would Recommend)": "Recommendation",
    "Sentiment (Negative / Neutral / Positive)":                               "Sentiment",
}
df_holdout = df_holdout.drop(columns=["Joy (0/1).1"], errors="ignore")
df_holdout = df_holdout.rename(columns={**emotion_rename, **eval_rename})

# Binarize eval dims across all splits
for dim, mapping in EVAL_BINARIZE.items():
    for df in [df_train_full, df_test, df_holdout]:
        if dim in df.columns:
            df[dim] = df[dim].map(mapping).fillna(0).astype(int)

print(f"Train full : {len(df_train_full)}")
print(f"Test       : {len(df_test)}")
print(f"Holdout    : {len(df_holdout)}")
print(f"Missing cols in holdout: {[c for c in CLASSES if c not in df_holdout.columns]}")

# Train/Val split — 90/10
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
    train_idx, val_idx = next(msss.split(df_train_full, df_train_full[EMOTIONS].values))
    df_train = df_train_full.iloc[train_idx].copy()
    df_val   = df_train_full.iloc[val_idx].copy()
    print("\nUsing: Iterative multi-label stratification")
except ImportError:
    from sklearn.model_selection import train_test_split
    df_train, df_val = train_test_split(df_train_full, test_size=0.1, random_state=42)
    print("\nUsing: Random split")

print(f"  Train      : {len(df_train)}")
print(f"  Validation : {len(df_val)}")

# Label arrays
train_labels   = df_train[CLASSES].values.astype(np.float32)
val_labels     = df_val[CLASSES].values.astype(np.float32)
test_labels    = df_test[CLASSES].values.astype(np.float32)
holdout_labels = df_holdout[CLASSES].values.astype(np.float32)

print(f"\nLabel distribution (train emotions):")
for i, cls in enumerate(EMOTIONS):
    count = train_labels[:, i].sum()
    print(f"  {cls:<15}: {int(count)} ({count/len(train_labels)*100:.1f}%)")

print("\n✓ Data ready")

LOADING DATA
Train full : 16328
Test       : 4083
Holdout    : 1441
Missing cols in holdout: []

Using: Iterative multi-label stratification
  Train      : 14695
  Validation : 1633

Label distribution (train emotions):
  Joy            : 6982 (47.5%)
  Love           : 2673 (18.2%)
  Surprise       : 852 (5.8%)
  Pride          : 229 (1.6%)
  Gratitude      : 693 (4.7%)
  Contentment    : 7624 (51.9%)
  Relief         : 1858 (12.6%)
  Hope           : 1611 (11.0%)
  Interest       : 1975 (13.4%)
  Anger          : 824 (5.6%)
  Frustration    : 4639 (31.6%)
  Disappointment : 5900 (40.1%)
  Sadness        : 78 (0.5%)
  Fear           : 241 (1.6%)
  Guilt          : 41 (0.3%)
  Disgust        : 674 (4.6%)

✓ Data ready


In [ ]:
# ── CELL 3: Evaluation functions (run once) ───────────────────────────────────
def evaluate_multilabel(y_true, y_pred, y_prob, class_names):
    per_class = {}
    for i, cls in enumerate(class_names):
        y_t, y_p, y_pr = y_true[:, i], y_pred[:, i], y_prob[:, i]
        try:
            auc = roc_auc_score(y_t, y_pr)
        except:
            auc = 0.0
        per_class[cls] = {
            'f1':        f1_score(y_t, y_p, zero_division=0),
            'precision': precision_score(y_t, y_p, zero_division=0),
            'recall':    recall_score(y_t, y_p, zero_division=0),
            'mcc':       matthews_corrcoef(y_t, y_p),
            'auc':       auc,
            'support':   int(y_t.sum()),
        }
    macro = {
        'f1_macro':        np.mean([per_class[c]['f1']        for c in class_names]),
        'precision_macro': np.mean([per_class[c]['precision'] for c in class_names]),
        'recall_macro':    np.mean([per_class[c]['recall']    for c in class_names]),
        'mcc_macro':       np.mean([per_class[c]['mcc']       for c in class_names]),
        'auc_macro':       np.mean([per_class[c]['auc']       for c in class_names]),
    }
    micro = {
        'f1_micro':        f1_score(y_true, y_pred, average='micro',   zero_division=0),
        'precision_micro': precision_score(y_true, y_pred, average='micro', zero_division=0),
        'recall_micro':    recall_score(y_true, y_pred, average='micro',    zero_division=0),
    }
    sample = {
        'f1_samples':      f1_score(y_true, y_pred, average='samples', zero_division=1),
        'jaccard_samples': jaccard_score(y_true, y_pred, average='samples', zero_division=1),
    }
    overall = {
        'exact_match_ratio': (y_pred == y_true).all(axis=1).mean(),
        'hamming_loss':      hamming_loss(y_true, y_pred),
    }
    return {'per_class': per_class, 'macro': macro, 'micro': micro,
            'sample': sample, 'overall': overall}

def print_report(metrics, class_names, config_name):
    print(f"\n{'='*70}")
    print(f"HOLDOUT EVALUATION — {config_name}")
    print(f"{'='*70}")
    print(f"\n{'Class':<16} {'F1':>7} {'AUC':>7} {'MCC':>7} {'Prec':>7} {'Rec':>7} {'Support':>8}")
    print("-"*62)
    for cls in class_names:
        m = metrics['per_class'][cls]
        print(f"{cls:<16} {m['f1']:>7.4f} {m['auc']:>7.4f} {m['mcc']:>7.4f} "
              f"{m['precision']:>7.4f} {m['recall']:>7.4f} {m['support']:>8}")
    print("-"*62)
    m = metrics['macro']
    print(f"{'Macro Avg':<16} {m['f1_macro']:>7.4f} {m['auc_macro']:>7.4f} "
          f"{m['mcc_macro']:>7.4f} {m['precision_macro']:>7.4f} {m['recall_macro']:>7.4f}")
    print(f"\n  Macro F1    : {metrics['macro']['f1_macro']:.4f}  ← Primary")
    print(f"  Macro AUC   : {metrics['macro']['auc_macro']:.4f}")
    print(f"  Macro MCC   : {metrics['macro']['mcc_macro']:.4f}")
    print(f"  Micro F1    : {metrics['micro']['f1_micro']:.4f}")
    print(f"  Exact Match : {metrics['overall']['exact_match_ratio']:.4f}")
    print(f"  Hamming Loss: {metrics['overall']['hamming_loss']:.4f}")

def save_results(metrics, trainer, config_name, model_name,
                 holdout_probs, holdout_preds, perf_path, hparams):
    df_preds = df_holdout.copy()
    for i, cls in enumerate(CLASSES):
        df_preds[f'{cls}_prob'] = holdout_probs[:, i]
        df_preds[f'{cls}_pred'] = holdout_preds[:, i]
    df_preds.to_csv(f"{perf_path}/holdout_predictions.csv", index=False)

    summary = {
        'config':          config_name,
        'model':           model_name,
        'macro':           metrics['macro'],
        'micro':           metrics['micro'],
        'sample':          metrics['sample'],
        'overall':         metrics['overall'],
        'per_class':       metrics['per_class'],
        'hyperparameters': hparams,
    }
    with open(f"{perf_path}/holdout_metrics.json", 'w') as f:
        json.dump(summary, f, indent=2, default=float)

    if trainer is not None:
        with open(f"{perf_path}/training_history.json", 'w') as f:
            json.dump(trainer.state.log_history, f, indent=2)

    print(f"✓ Saved to {perf_path}")

def get_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"Device: CUDA ({torch.cuda.get_device_name(0)})")
        print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        return device, True, False
    else:
        print("Device: CPU")
        return torch.device("cpu"), False, False

print("✓ Evaluation functions ready")

✓ Evaluation functions ready


In [ ]:
# ── CELL 4: Tokenize (run once per MAX_LENGTH change) ─────────────────────────
MAX_LENGTH = 128  # start here — fast, covers median review length

tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

def tokenize(texts, labels):
    enc = tokenizer(
        texts.tolist(),
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
        return_tensors=None,
    )
    return Dataset.from_dict({
        'input_ids':      enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'labels':         labels.tolist(),
    })

print(f"Tokenizing at MAX_LENGTH={MAX_LENGTH}...")
train_dataset   = tokenize(df_train['review_text'].values,   train_labels)
val_dataset     = tokenize(df_val['review_text'].values,     val_labels)
test_dataset    = tokenize(df_test['review_text'].values,    test_labels)
holdout_dataset = tokenize(df_holdout['review_text'].values, holdout_labels)

lengths = [len(ids) for ids in train_dataset['input_ids']]
truncated = sum(1 for l in lengths if l == MAX_LENGTH)
print(f"  Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"  Truncated: {truncated}/{len(lengths)} ({truncated/len(lengths)*100:.1f}%)")
print("✓ Tokenization done")

Tokenizing at MAX_LENGTH=128...
  Train: 14695 | Val: 1633
  Truncated: 14695/14695 (100.0%)
✓ Tokenization done


In [ ]:
# ── CELL 5: Train + Evaluate one config ───────────────────────────────────────
def predict(dataset, model, device, batch_size=64):
    from torch.utils.data import DataLoader
    loader    = DataLoader(dataset, batch_size=batch_size)
    all_probs = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            input_ids      = torch.stack(batch['input_ids'], dim=1).to(device)
            attention_mask = torch.stack(batch['attention_mask'], dim=1).to(device)
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            probs          = torch.sigmoid(outputs.logits).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)

def run_config(config_name, model_name, learning_rate, batch_size,
               gradient_accumulation, num_epochs, threshold=0.3, seed=42):

    print(f"\n{'='*60}")
    print(f"CONFIG: {config_name}")
    print(f"{'='*60}")
    print(f"  Model      : {model_name}")
    print(f"  LR         : {learning_rate}")
    print(f"  Batch      : {batch_size} x {gradient_accumulation} = {batch_size*gradient_accumulation}")
    print(f"  Epochs     : {num_epochs}")
    print(f"  Max length : {MAX_LENGTH}")
    print(f"  Threshold  : {threshold}")

    device, use_fp16, use_bf16 = get_device()

    save_path = f"{OUTPUT_DIR}/roberta_models/{config_name}/best_model"
    perf_path = f"{OUTPUT_DIR}/roberta_models/{config_name}/performance"
    os.makedirs(save_path, exist_ok=True)
    os.makedirs(perf_path, exist_ok=True)

    model = RobertaForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS,
        problem_type="multi_label_classification",
    )
    model.to(device)

    args = TrainingArguments(
        output_dir=f"{OUTPUT_DIR}/roberta_models/{config_name}",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_ratio=0.1,
        optim="adamw_torch",
        lr_scheduler_type="linear",
        max_grad_norm=1.0,
        eval_strategy="no",
        save_strategy="no",
        logging_steps=50,
        logging_first_step=True,
        report_to="none",
        fp16=use_fp16,
        bf16=use_bf16,
        seed=seed,
        data_seed=seed,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        remove_unused_columns=False,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        callbacks=[],
    )

    print(f"\n{'='*60}")
    print(f"TRAINING")
    print(f"{'='*60}")

    train_result = trainer.train()
    print(f"\n  Runtime    : {train_result.metrics['train_runtime']:.0f}s "
          f"({train_result.metrics['train_runtime']/3600:.1f}hrs)")
    print(f"  Final loss : {train_result.metrics['train_loss']:.4f}")

    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"  Model saved: {save_path}")

    print("\nRunning holdout predictions...")
    holdout_probs = predict(holdout_dataset, model, device)
    holdout_preds = (holdout_probs >= threshold).astype(int)

    metrics = evaluate_multilabel(holdout_labels, holdout_preds, holdout_probs, CLASSES)
    print_report(metrics, CLASSES, config_name)

    hparams = {
        'learning_rate':         learning_rate,
        'batch_size':            batch_size,
        'gradient_accumulation': gradient_accumulation,
        'num_epochs':            num_epochs,
        'max_length':            MAX_LENGTH,
        'threshold':             threshold,
    }
    save_results(metrics, trainer, config_name, model_name,
                 holdout_probs, holdout_preds, perf_path, hparams)

    del model
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  GPU cleared for next config")

    return metrics['macro']['f1_macro']

print("✓ run_config() ready")

✓ run_config() ready


In [ ]:
# ── CELL 6: EXECUTE all 4 configs ────────────────────────────────────────────
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

results = {}

results["config1_baseline"] = run_config(
    config_name          = "config1_baseline",
    model_name           = "roberta-large",
    learning_rate        = 2e-5,
    batch_size           = 8,
    gradient_accumulation= 8,
    num_epochs           = 4,
)

results["config2_higher_lr"] = run_config(
    config_name          = "config2_higher_lr",
    model_name           = "roberta-large",
    learning_rate        = 5e-5,
    batch_size           = 8,
    gradient_accumulation= 8,
    num_epochs           = 4,
)

results["config3_roberta_base"] = run_config(
    config_name          = "config3_roberta_base",
    model_name           = "roberta-base",
    learning_rate        = 2e-5,
    batch_size           = 32,
    gradient_accumulation= 2,
    num_epochs           = 4,
)

MAX_LENGTH = 256
print(f"\nRetokenizing at MAX_LENGTH={MAX_LENGTH}...")
train_dataset   = tokenize(df_train['review_text'].values,   train_labels)
val_dataset     = tokenize(df_val['review_text'].values,     val_labels)
test_dataset    = tokenize(df_test['review_text'].values,    test_labels)
holdout_dataset = tokenize(df_holdout['review_text'].values, holdout_labels)
print("✓ Retokenized")

results["config4_long_context"] = run_config(
    config_name          = "config4_long_context",
    model_name           = "roberta-large",
    learning_rate        = 2e-5,
    batch_size           = 4,
    gradient_accumulation= 16,
    num_epochs           = 4,
)

print(f"\n{'='*60}")
print("STEP 9 COMPLETE — CONFIG COMPARISON")
print(f"{'='*60}")
print(f"{'Config':<30} {'Macro F1':>10}")
print("-"*42)
for config, f1 in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{config:<30} {f1:>10.4f}")
best = max(results, key=results.get)
print(f"\n✓ Best config: {best} (F1 = {results[best]:.4f})")


CONFIG: config1_baseline
  Model      : roberta-large
  LR         : 2e-05
  Batch      : 8 x 8 = 64
  Epochs     : 4
  Max length : 128
  Threshold  : 0.3
Device: CUDA (Tesla T4)
  VRAM: 15.6 GB


RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



TRAINING


Step,Training Loss
1,5.232838
50,4.243469
100,2.514660
150,1.981541
200,1.787159
250,1.626165
300,1.549479
350,1.465657
400,1.393103
450,1.408060



  Runtime    : 1380s (0.4hrs)
  Final loss : 1.5386


...

RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



TRAINING


Step,Training Loss
1,10.484176
50,8.522700
100,4.998038
150,3.835191
200,3.441964
250,3.061610
300,2.866189
350,2.743354
400,2.547284
450,2.554313



  Runtime    : 2692s (0.7hrs)
  Final loss : 2.8922


  Model saved: data/roberta_models/config4_long_context/best_model

Running holdout predictions...

HOLDOUT EVALUATION — config4_long_context

Class                 F1     AUC     MCC    Prec     Rec  Support
--------------------------------------------------------------
Joy               0.7493  0.9142  0.5884  0.6073  0.9781      547
Love              0.7552  0.9379  0.6810  0.7877  0.7252      353
Surprise          0.4133  0.8267  0.4406  0.9101  0.2673      303
Pride             0.0000  0.8229  0.0000  0.0000  0.0000       31
Gratitude         0.4535  0.8748  0.4394  0.6190  0.3578      109
Contentment       0.9475  0.9869  0.8488  0.9773  0.9196      982
Relief            0.5942  0.9083  0.5459  0.5068  0.7179      156
Hope              0.4737  0.9375  0.4987  0.3273  0.8571       63
Interest          0.5450  0.8621  0.4719  0.5590  0.5317      205
Anger             0.6919  0.9739  0.6710  0.6348  0.7604       96
Frustration       0.7294  0.9098  0.6202  0.6955  0.7669      399
Di

In [ ]:
# ── Find and recover config 1 ─────────────────────────────────────────────────
import glob
from torch.utils.data import DataLoader

config_name = "config1_baseline"
base_path   = f"{OUTPUT_DIR}/roberta_models/{config_name}"
perf_path   = f"{base_path}/performance"
os.makedirs(perf_path, exist_ok=True)

# Find best checkpoint
checkpoints = sorted(glob.glob(f"{base_path}/checkpoint-*"))
print(f"Found checkpoints: {checkpoints}")

if not checkpoints:
    print("No checkpoints found — need to rerun config 1")
else:
    # Load from last checkpoint (all 4 epochs ran so last = best)
    load_path = checkpoints[-1]
    print(f"Loading from: {load_path}")

    device, use_fp16, use_bf16 = get_device()

    model     = RobertaForSequenceClassification.from_pretrained(load_path)
    tokenizer = RobertaTokenizer.from_pretrained(load_path)
    model.to(device)
    model.eval()

    def predict(dataset, model, device, batch_size=64):
        from torch.utils.data import DataLoader
        loader    = DataLoader(dataset, batch_size=batch_size)
        all_probs = []
        model.eval()
        with torch.no_grad():
            for batch in loader:
                input_ids      = torch.stack(batch['input_ids'], dim=1).to(device)
                attention_mask = torch.stack(batch['attention_mask'], dim=1).to(device)
                outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
                probs          = torch.sigmoid(outputs.logits).cpu().numpy()
                all_probs.append(probs)
        return np.vstack(all_probs)

    # Re-run from here
    print("Running holdout predictions...")
    holdout_probs = predict(holdout_dataset, model, device)
    holdout_preds = (holdout_probs >= 0.5).astype(int)

    metrics = evaluate_multilabel(holdout_labels, holdout_preds, holdout_probs, CLASSES)
    print_report(metrics, CLASSES, "config1_baseline")

    hparams = {
        'learning_rate': 2e-5, 'batch_size': 32,
        'gradient_accumulation': 2, 'num_epochs': 4,
        'max_length': 128, 'threshold': 0.5,
    }
    save_results(metrics, None, "config1_baseline", "roberta-large",
                holdout_probs, holdout_preds,
                f"{OUTPUT_DIR}/roberta_models/config1_baseline/performance", hparams)

    results["config1_baseline"] = metrics['macro']['f1_macro']
    print(f"\n✓ Config 1 recovered: Macro F1 = {metrics['macro']['f1_macro']:.4f}")

Found checkpoints: ['data/roberta_models/config1_baseline/checkpoint-690', 'data/roberta_models/config1_baseline/checkpoint-920']
Loading from: data/roberta_models/config1_baseline/checkpoint-920
Device: CUDA (Tesla T4)
  VRAM: 15.6 GB


Running holdout predictions...

HOLDOUT EVALUATION — config1_baseline

Class                 F1     AUC     MCC    Prec     Rec  Support
--------------------------------------------------------------
Joy               0.7597  0.9094  0.6000  0.6351  0.9452      547
Love              0.7127  0.9403  0.6445  0.8222  0.6289      353
Surprise          0.3467  0.8022  0.3897  0.9028  0.2145      303
Pride             0.0000  0.7905  0.0000  0.0000  0.0000       31
Gratitude         0.2482  0.8884  0.2830  0.6071  0.1560      109
Contentment       0.9225  0.9814  0.7989  0.9861  0.8666      982
Relief            0.5270  0.9042  0.4739  0.5571  0.5000      156
Hope              0.4533  0.9112  0.4303  0.3908  0.5397       63
Interest          0.4532  0.8624  0.4014  0.5952  0.3659      205
Anger             0.6667  0.9688  0.6494  0.7436  0.6042       96
Frustration       0.7161  0.8998  0.6141  0.7453  0.6892      399
Disappointment    0.8451  0.9551  0.7652  0.9266  0.7768      569
Sadness 

In [ ]:
# ── Threshold tuning on config 1 results ─────────────────────────────────────
for thresh in [0.3, 0.35, 0.4, 0.45, 0.5]:
    preds   = (holdout_probs >= thresh).astype(int)
    metrics_t = evaluate_multilabel(holdout_labels, preds, holdout_probs, CLASSES)
    print(f"Threshold {thresh}: Macro F1 = {metrics_t['macro']['f1_macro']:.4f} | "
          f"Micro F1 = {metrics_t['micro']['f1_micro']:.4f}")

Threshold 0.3: Macro F1 = 0.5590 | Micro F1 = 0.7938
Threshold 0.35: Macro F1 = 0.5542 | Micro F1 = 0.7904
Threshold 0.4: Macro F1 = 0.5492 | Micro F1 = 0.7871
Threshold 0.45: Macro F1 = 0.5408 | Micro F1 = 0.7817
Threshold 0.5: Macro F1 = 0.5364 | Micro F1 = 0.7781
